In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('dataset-6_for_model.csv')
df.shape

(83094, 5)

In [2]:
df.head()

,Genome Name,Species,Antibiotic,Target,Species_Mapped
0,Escherichia coli strain UMDUCS23,strain UMDUCS23,trimethoprim/sulfamethoxazole,0,Collection: UMDUC
1,Escherichia coli O23:H16 strain ECO0238,O23:H16 strain ECO0238,tetracycline,1,Serotype: O:H Type
2,Escherichia coli strain 36549,strain 36549,cefalothin,0,Numbered Isolate
3,Escherichia coli PNUSAE020616,PNUSAE020616,norfloxacin,0,Collection: CDC/PNUSAE
4,Escherichia coli strain 113,strain 113,doxycycline,1,Numbered Isolate


In [3]:
print(df['Target'].value_counts())

Target
0    57911
1    25183
Name: count, dtype: int64


In [4]:
print(df.columns.tolist())

['Genome Name', 'Species', 'Antibiotic', 'Target', 'Species_Mapped']


In [5]:
df = df.drop(columns=['Genome Name', 'Species'])

In [6]:
df.head()

,Antibiotic,Target,Species_Mapped
0,trimethoprim/sulfamethoxazole,0,Collection: UMDUC
1,tetracycline,1,Serotype: O:H Type
2,cefalothin,0,Numbered Isolate
3,norfloxacin,0,Collection: CDC/PNUSAE
4,doxycycline,1,Numbered Isolate


In [7]:
X = df[['Antibiotic', 'Species_Mapped']].copy()
y = df['Target']

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 80-20 split
    random_state=42,      # reproducible
    stratify=y            # keep class ratio same
)

In [11]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

# Fit on training set only
X_train_encoded = encoder.fit_transform(X_train)
# Apply to test set
X_test_encoded = encoder.transform(X_test)

In [12]:
import numpy as np

neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos

print("Scale positive weight:", scale_pos_weight)

Scale positive weight: 2.299662464012707


In [13]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight='balanced',  # automatically handles imbalance
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_encoded, y_train)

,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [16]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,  # handle imbalance
    random_state=42,
    eval_metric='logloss'
)

xgb.fit(X_train_encoded, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [17]:
from sklearn.metrics import classification_report, roc_auc_score

# Random Forest
y_pred_rf = rf.predict(X_test_encoded)
rf_probs = rf.predict_proba(X_test_encoded)[:, 1]

print("Random Forest Report:\n", classification_report(y_test, y_pred_rf))
print("Random Forest ROC-AUC:", roc_auc_score(y_test, rf_probs))

# XGBoost
y_pred_xgb = xgb.predict(X_test_encoded)
xgb_probs = xgb.predict_proba(X_test_encoded)[:, 1]

print("XGBoost Report:\n", classification_report(y_test, y_pred_xgb))
print("XGBoost ROC-AUC:", roc_auc_score(y_test, xgb_probs))

Random Forest Report:
               precision    recall  f1-score   support

           0       0.86      0.63      0.73     11582
           1       0.47      0.77      0.59      5037

    accuracy                           0.67     16619
   macro avg       0.67      0.70      0.66     16619
weighted avg       0.74      0.67      0.68     16619

Random Forest ROC-AUC: 0.7658462758080278
XGBoost Report:
               precision    recall  f1-score   support

           0       0.86      0.62      0.72     11582
           1       0.47      0.77      0.58      5037

    accuracy                           0.67     16619
   macro avg       0.66      0.69      0.65     16619
weighted avg       0.74      0.67      0.68     16619

XGBoost ROC-AUC: 0.7621838766123261


In [18]:
X = df[['Antibiotic', 'Species_Mapped']].copy()
y = df['Target']

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [22]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

# Ensure X_train and X_test are only categorical columns
# e.g., X_train = df[['Antibiotic', 'Species_Mapped']]

# Use sparse=True for memory efficiency (especially if many categories)
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit on train and transform
X_train_ohe = encoder.fit_transform(X_train)
X_test_ohe = encoder.transform(X_test)

# Convert back to DataFrame with proper column names
ohe_columns = encoder.get_feature_names_out(X_train.columns)
X_train_ohe = pd.DataFrame(X_train_ohe, columns=ohe_columns, index=X_train.index)
X_test_ohe = pd.DataFrame(X_test_ohe, columns=ohe_columns, index=X_test.index)

print("X_train_ohe shape:", X_train_ohe.shape)
print("X_test_ohe shape:", X_test_ohe.shape)

X_train_ohe shape: (66475, 117)
X_test_ohe shape: (16619, 117)


In [23]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_res, y_train_res = smote.fit_resample(X_train_ohe, y_train)

import pandas as pd
print(pd.Series(y_train_res).value_counts())

Target
0    46329
1    46329
Name: count, dtype: int64


In [24]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_res, y_train_res)

,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [25]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

xgb.fit(X_train_res, y_train_res)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [26]:
from sklearn.metrics import classification_report, roc_auc_score

# Random Forest
y_pred_rf = rf.predict(X_test_ohe)
rf_probs = rf.predict_proba(X_test_ohe)[:, 1]

print("Random Forest Report:\n", classification_report(y_test, y_pred_rf))
print("Random Forest ROC-AUC:", roc_auc_score(y_test, rf_probs))

# XGBoost
y_pred_xgb = xgb.predict(X_test_ohe)
xgb_probs = xgb.predict_proba(X_test_ohe)[:, 1]

print("XGBoost Report:\n", classification_report(y_test, y_pred_xgb))
print("XGBoost ROC-AUC:", roc_auc_score(y_test, xgb_probs))

Random Forest Report:
               precision    recall  f1-score   support

           0       0.86      0.63      0.73     11582
           1       0.47      0.77      0.59      5037

    accuracy                           0.67     16619
   macro avg       0.67      0.70      0.66     16619
weighted avg       0.74      0.67      0.68     16619

Random Forest ROC-AUC: 0.7655768484000643
XGBoost Report:
               precision    recall  f1-score   support

           0       0.87      0.62      0.72     11582
           1       0.47      0.78      0.59      5037

    accuracy                           0.67     16619
   macro avg       0.67      0.70      0.66     16619
weighted avg       0.75      0.67      0.68     16619

XGBoost ROC-AUC: 0.7666073422414077


In [29]:
X = df[['Antibiotic', 'Species_Mapped']].copy()
y = df['Target']

# Create a dummy numeric feature (needed for SMOTENC)
# Use random numbers or a small noise around label encoding
import numpy as np

X['NumericFeature'] = np.random.rand(len(X))

In [30]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [31]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

X_train[['Antibiotic', 'Species_Mapped']] = encoder.fit_transform(
    X_train[['Antibiotic', 'Species_Mapped']]
)
X_test[['Antibiotic', 'Species_Mapped']] = encoder.transform(
    X_test[['Antibiotic', 'Species_Mapped']]
)

In [32]:
from imblearn.over_sampling import SMOTENC

# categorical columns are 0 and 1 (Antibiotic, Species_Mapped)
smote_nc = SMOTENC(categorical_features=[0, 1], random_state=42)

X_train_res, y_train_res = smote_nc.fit_resample(X_train, y_train)

import pandas as pd
print(pd.Series(y_train_res).value_counts())

Target
0    46329
1    46329
Name: count, dtype: int64


In [33]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train_res, y_train_res)

xgb = XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss')
xgb.fit(X_train_res, y_train_res)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [34]:
from sklearn.metrics import classification_report, roc_auc_score

# RF
y_pred_rf = rf.predict(X_test)
rf_probs = rf.predict_proba(X_test)[:, 1]
print("RF Report:\n", classification_report(y_test, y_pred_rf))
print("RF ROC-AUC:", roc_auc_score(y_test, rf_probs))

# XGB
y_pred_xgb = xgb.predict(X_test)
xgb_probs = xgb.predict_proba(X_test)[:, 1]
print("XGB Report:\n", classification_report(y_test, y_pred_xgb))
print("XGB ROC-AUC:", roc_auc_score(y_test, xgb_probs))

RF Report:
               precision    recall  f1-score   support

           0       0.77      0.68      0.72     11582
           1       0.41      0.52      0.46      5037

    accuracy                           0.63     16619
   macro avg       0.59      0.60      0.59     16619
weighted avg       0.66      0.63      0.64     16619

RF ROC-AUC: 0.6691855678101202
XGB Report:
               precision    recall  f1-score   support

           0       0.85      0.65      0.73     11582
           1       0.47      0.73      0.58      5037

    accuracy                           0.67     16619
   macro avg       0.66      0.69      0.66     16619
weighted avg       0.74      0.67      0.69     16619

XGB ROC-AUC: 0.7557028258543487


In [35]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

In [37]:
from sklearn.preprocessing import OneHotEncoder

# Features and target
X = df[['Antibiotic', 'Species_Mapped']].copy()
y = df['Target']

# One-Hot Encoding (updated parameter name)
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit and transform
X_ohe = encoder.fit_transform(X)

X_ohe = pd.DataFrame(X_ohe, columns=encoder.get_feature_names_out(X.columns), index=X.index)

print("Shape after OHE:", X_ohe.shape)

Shape after OHE: (83094, 118)


In [38]:
X_train, X_test, y_train, y_test = train_test_split(
    X_ohe, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [39]:
from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = dict(enumerate(class_weights))
print(class_weights_dict)  # e.g., {0: 0.68, 1: 1.57}

{0: 0.7174232122428716, 1: 1.6498312320063535}


In [41]:
model = Sequential()
model.add(Dense(64,activation='relu',input_dim=118))
model.add(Dense(32,activation='relu'))
model.add(Dense(1,activation='sigmoid'))
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                7616      
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 9,729
Trainable params: 9,729
Non-trainable params: 0
_________________________________________________________________


In [42]:
model.compile(optimizer='Adam',loss='binary_crossentropy',metrics=['accuracy'])

In [43]:
history = model.fit(X_train,y_train,batch_size=32,epochs=5,validation_data=(X_test,y_test))

Epoch 1/5
2078/2078 [==============================] - 10s 4ms/step - loss: 0.5220 - accuracy: 0.7246 - val_loss: 0.5132 - val_accuracy: 0.7296
Epoch 2/5
2078/2078 [==============================] - 8s 4ms/step - loss: 0.5098 - accuracy: 0.7323 - val_loss: 0.5093 - val_accuracy: 0.7329
Epoch 3/5
2078/2078 [==============================] - 8s 4ms/step - loss: 0.5082 - accuracy: 0.7335 - val_loss: 0.5094 - val_accuracy: 0.7309
Epoch 4/5
2078/2078 [==============================] - 8s 4ms/step - loss: 0.5072 - accuracy: 0.7334 - val_loss: 0.5087 - val_accuracy: 0.7326
Epoch 5/5
2078/2078 [==============================] - 8s 4ms/step - loss: 0.5066 - accuracy: 0.7347 - val_loss: 0.5080 - val_accuracy: 0.7327


In [44]:
# Predictions
y_pred_prob = model.predict(X_test).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)

from sklearn.metrics import classification_report, roc_auc_score

print("MLP Classification Report:\n", classification_report(y_test, y_pred))
print("MLP ROC-AUC:", roc_auc_score(y_test, y_pred_prob))

520/520 [==============================] - 1s 2ms/step
MLP Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.89      0.82     11582
           1       0.59      0.37      0.46      5037

    accuracy                           0.73     16619
   macro avg       0.68      0.63      0.64     16619
weighted avg       0.71      0.73      0.71     16619

MLP ROC-AUC: 0.7694098209598478


In [45]:
model = Sequential()
model.add(Dense(32,activation='relu',input_dim=118))
model.add(Dense(1,activation='sigmoid'))
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 32)                3808      
                                                                 
 dense_4 (Dense)             (None, 1)                 33        
                                                                 
Total params: 3,841
Trainable params: 3,841
Non-trainable params: 0
_________________________________________________________________


In [46]:
model.compile(optimizer='Adam',loss='binary_crossentropy',metrics=['accuracy'])

In [47]:
history = model.fit(X_train,y_train,batch_size=32,epochs=5,validation_data=(X_test,y_test))

Epoch 1/5
2078/2078 [==============================] - 8s 4ms/step - loss: 0.5290 - accuracy: 0.7230 - val_loss: 0.5142 - val_accuracy: 0.7286
Epoch 2/5
2078/2078 [==============================] - 7s 4ms/step - loss: 0.5113 - accuracy: 0.7310 - val_loss: 0.5109 - val_accuracy: 0.7331
Epoch 3/5
2078/2078 [==============================] - 7s 3ms/step - loss: 0.5094 - accuracy: 0.7327 - val_loss: 0.5090 - val_accuracy: 0.7345
Epoch 4/5
2078/2078 [==============================] - 8s 4ms/step - loss: 0.5080 - accuracy: 0.7338 - val_loss: 0.5100 - val_accuracy: 0.7339
Epoch 5/5
2078/2078 [==============================] - 8s 4ms/step - loss: 0.5075 - accuracy: 0.7338 - val_loss: 0.5092 - val_accuracy: 0.7333


In [48]:
# Predictions
y_pred_prob = model.predict(X_test).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)

from sklearn.metrics import classification_report, roc_auc_score

print("MLP Classification Report:\n", classification_report(y_test, y_pred))
print("MLP ROC-AUC:", roc_auc_score(y_test, y_pred_prob))

520/520 [==============================] - 1s 1ms/step
MLP Classification Report:
               precision    recall  f1-score   support

           0       0.76      0.90      0.82     11582
           1       0.60      0.36      0.45      5037

    accuracy                           0.73     16619
   macro avg       0.68      0.63      0.64     16619
weighted avg       0.71      0.73      0.71     16619

MLP ROC-AUC: 0.7677035387965012


In [50]:
model = Sequential()
model.add(Dense(128,activation='relu',input_dim=118))
model.add(Dropout(0.3))
model.add(Dense(64,activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(32,activation='tanh'))
model.add(Dropout(0.2))
model.add(Dense(1,activation='sigmoid'))
model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_8 (Dense)             (None, 128)               15232     
                                                                 
 dropout_2 (Dropout)         (None, 128)               0         
                                                                 
 dense_9 (Dense)             (None, 64)                8256      
                                                                 
 dropout_3 (Dropout)         (None, 64)                0         
                                                                 
 dense_10 (Dense)            (None, 32)                2080      
                                                                 
 dropout_4 (Dropout)         (None, 32)                0         
                                                                 
 dense_11 (Dense)            (None, 1)                

In [53]:
model.compile(optimizer='Adam',loss='binary_crossentropy',metrics=['accuracy'])

In [54]:
history = model.fit(X_train,y_train,batch_size=32,epochs=5,validation_data=(X_test,y_test))

Epoch 1/5
2078/2078 [==============================] - 11s 5ms/step - loss: 0.5270 - accuracy: 0.7208 - val_loss: 0.5126 - val_accuracy: 0.7300
Epoch 2/5
2078/2078 [==============================] - 9s 4ms/step - loss: 0.5153 - accuracy: 0.7307 - val_loss: 0.5113 - val_accuracy: 0.7313
Epoch 3/5
2078/2078 [==============================] - 10s 5ms/step - loss: 0.5130 - accuracy: 0.7321 - val_loss: 0.5101 - val_accuracy: 0.7326
Epoch 4/5
2078/2078 [==============================] - 10s 5ms/step - loss: 0.5118 - accuracy: 0.7326 - val_loss: 0.5091 - val_accuracy: 0.7265
Epoch 5/5
2078/2078 [==============================] - 10s 5ms/step - loss: 0.5108 - accuracy: 0.7322 - val_loss: 0.5098 - val_accuracy: 0.7321


In [55]:
# Predictions
y_pred_prob = model.predict(X_test).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)

from sklearn.metrics import classification_report, roc_auc_score

print("MLP Classification Report:\n", classification_report(y_test, y_pred))
print("MLP ROC-AUC:", roc_auc_score(y_test, y_pred_prob))

520/520 [==============================] - 1s 1ms/step
MLP Classification Report:
               precision    recall  f1-score   support

           0       0.76      0.90      0.82     11582
           1       0.60      0.35      0.44      5037

    accuracy                           0.73     16619
   macro avg       0.68      0.62      0.63     16619
weighted avg       0.71      0.73      0.71     16619

MLP ROC-AUC: 0.767589703231144


In [56]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Dropout
import keras_tuner as kt

In [58]:
def build_model(hp):
    model = Sequential()
    counter = 0
    for i in range(hp.Int('num_layers',min_value=1,max_value=3)):
        if counter == 0:
            model.add(
                Dense(
                    hp.Int('units' + str(i), min_value=8,max_value=128,step=8),
                    activation= hp.Choice('activation'+str(i),values=['relu','tanh','sigmoid']),
                    input_dim=118
                )
            )
            model.add(
                Dropout(hp.Choice('dropout'+str(i), values=[0.1,0.2,0.3,0.4,0.5]))
            )
        else:
            model.add(
                Dense(
                    hp.Int('units' + str(i), min_value=8,max_value=128,step=8),
                    activation= hp.Choice('activation'+str(i),values=['relu','tanh','sigmoid'])
                )
            )
            model.add(
                Dropout(hp.Choice('dropout'+str(i), values=[0.1,0.2,0.3,0.4,0.5]))
            )
        counter+=1
    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer=hp.Choice('optimizer',values=['rmsprop','adam','sgd','nadam','adadelta','adamw']),
                 loss='binary_crossentropy',
                 metrics=['accuracy'])
    return model

In [59]:
tuner = kt.RandomSearch(build_model,
                       objective='val_accuracy',
                       max_trials=15,
                       directory='mydir',
                       project_name='project8')

In [60]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 15 Complete [00h 00m 48s]
val_accuracy: 0.6969131827354431

Best val_accuracy So Far: 0.7336783409118652
Total elapsed time: 00h 12m 41s


In [61]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 1,
 'units0': 96,
 'activation0': 'relu',
 'dropout0': 0.4,
 'optimizer': 'adam',
 'units1': 128,
 'activation1': 'sigmoid',
 'dropout1': 0.2,
 'units2': 80,
 'activation2': 'sigmoid',
 'dropout2': 0.1}

In [64]:
model = tuner.get_best_models(num_models=1)[0]
history = model.fit(X_train,y_train,epochs=15,initial_epoch=6,validation_data=(X_test, y_test))

Epoch 7/15
2078/2078 [==============================] - 8s 4ms/step - loss: 0.5086 - accuracy: 0.7344 - val_loss: 0.5080 - val_accuracy: 0.7340
Epoch 8/15
2078/2078 [==============================] - 8s 4ms/step - loss: 0.5087 - accuracy: 0.7336 - val_loss: 0.5078 - val_accuracy: 0.7348
Epoch 9/15
2078/2078 [==============================] - 11s 5ms/step - loss: 0.5085 - accuracy: 0.7331 - val_loss: 0.5082 - val_accuracy: 0.7322
Epoch 10/15
2078/2078 [==============================] - 9s 4ms/step - loss: 0.5085 - accuracy: 0.7338 - val_loss: 0.5081 - val_accuracy: 0.7336
Epoch 11/15
2078/2078 [==============================] - 8s 4ms/step - loss: 0.5081 - accuracy: 0.7333 - val_loss: 0.5078 - val_accuracy: 0.7322
Epoch 12/15
2078/2078 [==============================] - 7s 4ms/step - loss: 0.5075 - accuracy: 0.7337 - val_loss: 0.5078 - val_accuracy: 0.7333
Epoch 13/15
2078/2078 [==============================] - 8s 4ms/step - loss: 0.5067 - accuracy: 0.7347 - val_loss: 0.5079 - val_accu

In [65]:
# Predictions
y_pred_prob = model.predict(X_test).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)

print("MLP Classification Report:\n", classification_report(y_test, y_pred))
print("MLP ROC-AUC:", roc_auc_score(y_test, y_pred_prob))

520/520 [==============================] - 1s 1ms/step
MLP Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.89      0.82     11582
           1       0.59      0.39      0.47      5037

    accuracy                           0.73     16619
   macro avg       0.68      0.64      0.65     16619
weighted avg       0.72      0.73      0.72     16619

MLP ROC-AUC: 0.769916467218734


In [67]:
print(pd.Series(y_train).value_counts())

Target
0    46329
1    20146
Name: count, dtype: int64


In [68]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 96)                11424     
                                                                 
 dropout (Dropout)           (None, 96)                0         
                                                                 
 dense_1 (Dense)             (None, 1)                 97        
                                                                 
Total params: 11,521
Trainable params: 11,521
Non-trainable params: 0
_________________________________________________________________


In [70]:
history = model.fit(X_train,y_train,epochs=20,initial_epoch=15,validation_data=(X_test, y_test),class_weight=class_weights_dict)

Epoch 16/20
2078/2078 [==============================] - 22s 11ms/step - loss: 0.5630 - accuracy: 0.6743 - val_loss: 0.5721 - val_accuracy: 0.6694
Epoch 17/20
2078/2078 [==============================] - 23s 11ms/step - loss: 0.5619 - accuracy: 0.6717 - val_loss: 0.5650 - val_accuracy: 0.6691
Epoch 18/20
2078/2078 [==============================] - 23s 11ms/step - loss: 0.5621 - accuracy: 0.6729 - val_loss: 0.5650 - val_accuracy: 0.6704
Epoch 19/20
2078/2078 [==============================] - 23s 11ms/step - loss: 0.5619 - accuracy: 0.6730 - val_loss: 0.5710 - val_accuracy: 0.6658
Epoch 20/20
2078/2078 [==============================] - 23s 11ms/step - loss: 0.5626 - accuracy: 0.6726 - val_loss: 0.5655 - val_accuracy: 0.6702


In [71]:
from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = dict(enumerate(class_weights))
print(class_weights_dict)  # e.g., {0: 0.68, 1: 1.57}

{0: 0.7174232122428716, 1: 1.6498312320063535}


In [72]:
model = Sequential()
model.add(Dense(128,activation='relu',input_dim=118))
model.add(Dropout(0.3))
model.add(Dense(64,activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(32,activation='tanh'))
model.add(Dropout(0.2))
model.add(Dense(1,activation='sigmoid'))
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_2 (Dense)             (None, 128)               15232     
                                                                 
 dropout_1 (Dropout)         (None, 128)               0         
                                                                 
 dense_3 (Dense)             (None, 64)                8256      
                                                                 
 dropout_2 (Dropout)         (None, 64)                0         
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dropout_3 (Dropout)         (None, 32)                0         
                                                                 
 dense_5 (Dense)             (None, 1)                

In [73]:
model.compile(optimizer='Adam',loss='binary_crossentropy',metrics=['accuracy'])

In [74]:
history = model.fit(X_train,y_train,batch_size=32,epochs=5,validation_data=(X_test,y_test),class_weight=class_weights_dict)

Epoch 1/5
2078/2078 [==============================] - 27s 12ms/step - loss: 0.5868 - accuracy: 0.6575 - val_loss: 0.5797 - val_accuracy: 0.6578
Epoch 2/5
2078/2078 [==============================] - 25s 12ms/step - loss: 0.5730 - accuracy: 0.6634 - val_loss: 0.5847 - val_accuracy: 0.6540
Epoch 3/5
2078/2078 [==============================] - 25s 12ms/step - loss: 0.5704 - accuracy: 0.6660 - val_loss: 0.5929 - val_accuracy: 0.6402
Epoch 4/5
2078/2078 [==============================] - 25s 12ms/step - loss: 0.5686 - accuracy: 0.6661 - val_loss: 0.5669 - val_accuracy: 0.6659
Epoch 5/5
2078/2078 [==============================] - 26s 13ms/step - loss: 0.5678 - accuracy: 0.6665 - val_loss: 0.5800 - val_accuracy: 0.6543


In [77]:
from sklearn.preprocessing import OneHotEncoder

# Features and target
X = df[['Antibiotic', 'Species_Mapped']].copy()
y = df['Target']

# One-Hot Encoding (updated parameter name)
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit and transform
X_ohe = encoder.fit_transform(X)

# Optional: convert to DataFrame for clarity
import pandas as pd
X_ohe = pd.DataFrame(X_ohe, columns=encoder.get_feature_names_out(X.columns), index=X.index)

print("Shape after OHE:", X_ohe.shape)

Shape after OHE: (83094, 118)


In [78]:
X_train, X_test, y_train, y_test = train_test_split(
    X_ohe, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [79]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

import pandas as pd
print(pd.Series(y_train_res).value_counts())

Target
0    46329
1    46329
Name: count, dtype: int64


In [80]:
model = Sequential()
model.add(Dense(128,activation='relu',input_dim=118))
model.add(Dropout(0.3))
model.add(Dense(64,activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(32,activation='tanh'))
model.add(Dropout(0.2))
model.add(Dense(1,activation='sigmoid'))
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_6 (Dense)             (None, 128)               15232     
                                                                 
 dropout_4 (Dropout)         (None, 128)               0         
                                                                 
 dense_7 (Dense)             (None, 64)                8256      
                                                                 
 dropout_5 (Dropout)         (None, 64)                0         
                                                                 
 dense_8 (Dense)             (None, 32)                2080      
                                                                 
 dropout_6 (Dropout)         (None, 32)                0         
                                                                 
 dense_9 (Dense)             (None, 1)                

In [81]:
model.compile(optimizer='Adam',loss='binary_crossentropy',metrics=['accuracy'])

In [82]:
history = model.fit(
    X_train_res, y_train_res,
    epochs=20,
    batch_size=128,
    validation_split=0.2,
    verbose=2
)

Epoch 1/20
580/580 - 9s - loss: 0.5680 - accuracy: 0.6880 - val_loss: 0.8051 - val_accuracy: 0.5520 - 9s/epoch - 15ms/step
Epoch 2/20
580/580 - 6s - loss: 0.5510 - accuracy: 0.7038 - val_loss: 0.7756 - val_accuracy: 0.5507 - 6s/epoch - 11ms/step
Epoch 3/20
580/580 - 7s - loss: 0.5482 - accuracy: 0.7049 - val_loss: 0.7528 - val_accuracy: 0.5944 - 7s/epoch - 11ms/step
Epoch 4/20
580/580 - 6s - loss: 0.5466 - accuracy: 0.7056 - val_loss: 0.7575 - val_accuracy: 0.5294 - 6s/epoch - 11ms/step
Epoch 5/20
580/580 - 7s - loss: 0.5451 - accuracy: 0.7064 - val_loss: 0.7630 - val_accuracy: 0.5642 - 7s/epoch - 12ms/step
Epoch 6/20
580/580 - 7s - loss: 0.5446 - accuracy: 0.7077 - val_loss: 0.7962 - val_accuracy: 0.5177 - 7s/epoch - 11ms/step
Epoch 7/20
580/580 - 6s - loss: 0.5438 - accuracy: 0.7073 - val_loss: 0.7746 - val_accuracy: 0.5256 - 6s/epoch - 11ms/step
Epoch 8/20
580/580 - 7s - loss: 0.5439 - accuracy: 0.7063 - val_loss: 0.7646 - val_accuracy: 0.5567 - 7s/epoch - 12ms/step
Epoch 9/20
580/5

In [84]:
y_pred_prob = model.predict(X_test).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)

print("MLP Classification Report:\n", classification_report(y_test, y_pred))
print("MLP ROC-AUC:", roc_auc_score(y_test, y_pred_prob))
from sklearn.metrics import f1_score

f1 = f1_score(y_test, y_pred)
print("F1-score:", f1)

520/520 [==============================] - 3s 5ms/step
MLP Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.82      0.81     11582
           1       0.55      0.50      0.52      5037

    accuracy                           0.73     16619
   macro avg       0.67      0.66      0.67     16619
weighted avg       0.72      0.73      0.72     16619

MLP ROC-AUC: 0.7683515924483122
F1-score: 0.5234171273599666


In [86]:
class_weights_dict = {0: 0.68, 1: 1.57}  # example
model.fit(X_train, y_train, epochs=5, batch_size=64, class_weight=class_weights_dict)

Epoch 1/5
1039/1039 [==============================] - 13s 11ms/step - loss: 0.5357 - accuracy: 0.6715
Epoch 2/5
1039/1039 [==============================] - 12s 11ms/step - loss: 0.5340 - accuracy: 0.6723
Epoch 3/5
1039/1039 [==============================] - 12s 12ms/step - loss: 0.5337 - accuracy: 0.6703
Epoch 4/5
1039/1039 [==============================] - 12s 12ms/step - loss: 0.5340 - accuracy: 0.6695
Epoch 5/5
1039/1039 [==============================] - 12s 11ms/step - loss: 0.5337 - accuracy: 0.6704


In [87]:
y_pred_prob = model.predict(X_test).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)

print("MLP Classification Report:\n", classification_report(y_test, y_pred))
print("MLP ROC-AUC:", roc_auc_score(y_test, y_pred_prob))

f1 = f1_score(y_test, y_pred)
print("F1-score:", f1)

520/520 [==============================] - 3s 5ms/step
MLP Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.61      0.72     11582
           1       0.47      0.79      0.59      5037

    accuracy                           0.67     16619
   macro avg       0.67      0.70      0.65     16619
weighted avg       0.75      0.67      0.68     16619

MLP ROC-AUC: 0.7682968773263997
F1-score: 0.5893002893819099


In [89]:
from sklearn.metrics import f1_score
import numpy as np

y_prob = model.predict(X_test).flatten()
thresholds = np.arange(0.1, 0.9, 0.05)

best_thresh = 0.5
best_f1 = 0
for t in thresholds:
    y_pred = (y_prob > t).astype(int)
    f1 = f1_score(y_test, y_pred)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print("Best threshold:", best_thresh, "Best F1:", best_f1)

520/520 [==============================] - 3s 5ms/step
Best threshold: 0.5000000000000001 Best F1: 0.5893002893819099


In [90]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Concatenate, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, roc_auc_score, f1_score

In [93]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd

# --------------------------
# Features and target
# --------------------------
X = df[['Antibiotic', 'Species_Mapped']].copy()
y = df['Target'].copy()

# --------------------------
# Train-test split first (avoid leakage)
# --------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Make copies to avoid SettingWithCopyWarning
X_train = X_train.copy()
X_test = X_test.copy()

# --------------------------
# Encode 'Antibiotic' column safely
# --------------------------
le_ab = LabelEncoder()
X_train['Antibiotic_enc'] = le_ab.fit_transform(X_train['Antibiotic'])

# For unseen categories in test, map to a special value (-1)
X_test['Antibiotic_enc'] = X_test['Antibiotic'].apply(
    lambda x: le_ab.transform([x])[0] if x in le_ab.classes_ else -1
)

# --------------------------
# Encode 'Species_Mapped' column safely
# --------------------------
le_sp = LabelEncoder()
X_train['Species_enc'] = le_sp.fit_transform(X_train['Species_Mapped'])

X_test['Species_enc'] = X_test['Species_Mapped'].apply(
    lambda x: le_sp.transform([x])[0] if x in le_sp.classes_ else -1
)

# --------------------------
# Get number of categories for embedding layers
# --------------------------
n_antibiotic = X_train['Antibiotic_enc'].nunique() + 1  # +1 for unseen category
n_species = X_train['Species_enc'].nunique() + 1       # +1 for unseen category

print(f"Number of Antibiotic categories (for embedding): {n_antibiotic}")
print(f"Number of Species categories (for embedding): {n_species}")

# --------------------------
# Now X_train[['Antibiotic_enc', 'Species_enc']] and X_test[...] are ready
# to feed into embedding layers of an MLP
# --------------------------

Number of Antibiotic categories (for embedding): 69
Number of Species categories (for embedding): 50


In [94]:
class_weights_array = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights_dict = dict(enumerate(class_weights_array))
print("Class weights:", class_weights_dict)

Class weights: {0: 0.7174232122428716, 1: 1.6498312320063535}


In [95]:
# Inputs
input_ant = Input(shape=(1,))
input_sp = Input(shape=(1,))

# Embeddings
emb_size = 8  # can tune

emb_ant = Embedding(input_dim=n_antibiotic, output_dim=emb_size, input_length=1)(input_ant)
emb_sp = Embedding(input_dim=n_species, output_dim=emb_size, input_length=1)(input_sp)

# Flatten embeddings
emb_ant_flat = Flatten()(emb_ant)
emb_sp_flat = Flatten()(emb_sp)

# Concatenate embeddings
x = Concatenate()([emb_ant_flat, emb_sp_flat])

# MLP on top
x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(32, activation='relu')(x)
x = Dropout(0.2)(x)
output = Dense(1, activation='sigmoid')(x)

# Model
model = Model(inputs=[input_ant, input_sp], outputs=output)
model.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 1)]          0           []                               
                                                                                                  
 input_2 (InputLayer)           [(None, 1)]          0           []                               
                                                                                                  
 embedding (Embedding)          (None, 1, 8)         552         ['input_1[0][0]']                
                                                                                                  
 embedding_1 (Embedding)        (None, 1, 8)         400         ['input_2[0][0]']                
                                                                                              

In [96]:
history = model.fit(
    [X_train['Antibiotic_enc'], X_train['Species_enc']],
    y_train,
    epochs=30,
    batch_size=128,
    validation_split=0.2,
    class_weight=class_weights_dict,  # Option 1 for balancing
    verbose=2
)

Epoch 1/30
416/416 - 9s - loss: 0.5985 - accuracy: 0.6655 - val_loss: 0.5759 - val_accuracy: 0.6652 - 9s/epoch - 21ms/step
Epoch 2/30
416/416 - 6s - loss: 0.5735 - accuracy: 0.6686 - val_loss: 0.5682 - val_accuracy: 0.6749 - 6s/epoch - 14ms/step
Epoch 3/30
416/416 - 6s - loss: 0.5705 - accuracy: 0.6704 - val_loss: 0.5871 - val_accuracy: 0.6657 - 6s/epoch - 14ms/step
Epoch 4/30
416/416 - 6s - loss: 0.5704 - accuracy: 0.6719 - val_loss: 0.5873 - val_accuracy: 0.6566 - 6s/epoch - 14ms/step
Epoch 5/30
416/416 - 6s - loss: 0.5693 - accuracy: 0.6745 - val_loss: 0.5675 - val_accuracy: 0.6681 - 6s/epoch - 14ms/step
Epoch 6/30
416/416 - 6s - loss: 0.5684 - accuracy: 0.6725 - val_loss: 0.5826 - val_accuracy: 0.6688 - 6s/epoch - 14ms/step
Epoch 7/30
416/416 - 6s - loss: 0.5675 - accuracy: 0.6756 - val_loss: 0.5766 - val_accuracy: 0.6728 - 6s/epoch - 14ms/step
Epoch 8/30
416/416 - 6s - loss: 0.5675 - accuracy: 0.6745 - val_loss: 0.5671 - val_accuracy: 0.6710 - 6s/epoch - 14ms/step
Epoch 9/30
416/4

In [97]:
# Predict
y_pred_prob = model.predict([X_test['Antibiotic_enc'], X_test['Species_enc']]).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)

# Metrics
print("Classification Report:\n", classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_prob))
print("F1-score:", f1_score(y_test, y_pred))

520/520 [==============================] - 1s 2ms/step
Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.63      0.73     11582
           1       0.47      0.76      0.59      5037

    accuracy                           0.67     16619
   macro avg       0.67      0.70      0.66     16619
weighted avg       0.74      0.67      0.69     16619

ROC-AUC: 0.7687016612381792
F1-score: 0.5855478565445823


In [9]:
from sklearn.preprocessing import LabelEncoder

le_ant = LabelEncoder()
le_spec = LabelEncoder()

X['Antibiotic'] = le_ant.fit_transform(X['Antibiotic'])
X['Species_Mapped'] = le_spec.fit_transform(X['Species_Mapped'])

In [10]:
X['Antibiotic']

0        67
1        63
2        12
3        49
4        33
         ..
83089    14
83090    16
83091    41
83092    15
83093    15
Name: Antibiotic, Length: 83094, dtype: int32

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [15]:
X_train

,Antibiotic,Species_Mapped
7280,1,10
52926,65,38
62349,63,42
79509,5,0
23262,25,0
...,...,...
41599,24,16
56012,29,0
24908,65,0
50567,61,42


In [16]:
y_train

7280     0
52926    0
62349    0
79509    1
23262    0
        ..
41599    0
56012    1
24908    0
50567    0
10536    0
Name: Target, Length: 66475, dtype: int64

In [18]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)

In [13]:
!pip install imblearn

  Using cached imblearn-0.0-py2.py3-none-any.whl.metadata (355 bytes)
Using cached imblearn-0.0-py2.py3-none-any.whl (1.9 kB)

   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   -------------------------- ------------- 2/3 [imblearn]
   ---------------------------------------- 3/3 [imblearn]

